# Lib imports

In [ ]:
from dotenv import load_dotenv

In [2]:
load_dotenv(override=True)

True

In [ ]:
import autoroot  # noqa
import nest_asyncio

# Third-party imports
from llama_index.core import PropertyGraphIndex, StorageContext
from llama_index.core.node_parser import SemanticSplitterNodeParser
from llama_index.core.prompts import PromptTemplate
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.graph_stores.memgraph import MemgraphPropertyGraphStore
from llama_index.llms.openrouter import OpenRouter
from llama_index.readers.wikipedia import WikipediaReader
from rich import print as rprint

import ogre_kg.kg_processors.extractors as kg_ext
import ogre_kg.kg_processors.entity_processing.entity_disambiguation as kg_disamb
import ogre_kg.kg_processors.entity_processing.entity_merger as kg_merger
import ogre_kg.kg_processors.entity_processing.entity_similarity_finders as kg_sim
import ogre_kg.kg_processors.graph_db_utils as kg_db_utils
import ogre_kg.kg_processors.retrievers.memgraph_retrievers as mg_ret

In [4]:
nest_asyncio.apply()

## OGRE-KG modules overview

This notebook exercises the core OGRE-KG pipeline. The modules imported above cover four areas:

| Module | Purpose |
|--------|---------|
| `ogre_kg.kg_processors.graph_db_utils` | Builds full-text / vector indices in the graph backend so keyword retrievers can run efficiently. |
| `ogre_kg.kg_processors.entity_processing.entity_similarity_finders` | Finds groups of *similar* entity nodes using cosine similarity on stored embeddings (Memgraph backend) or fuzzy string matching (backend-agnostic). |
| `ogre_kg.kg_processors.entity_processing.entity_disambiguation` | Composes a similarity finder + a merger into a single `process()` call that runs the full find-then-merge pipeline. |
| `ogre_kg.kg_processors.entity_processing.entity_merger` | Applies the actual merge/synonym-creation strategy in the graph store (e.g. Memgraph `refactor.merge_nodes`, or Neo4j APOC). |
| `ogre_kg.kg_processors.retrievers.memgraph_retrievers` | Custom LlamaIndex retrievers that use Memgraph full-text search to seed context retrieval from the knowledge graph. |

# LLM config

In [ ]:
llm = OpenRouter(model="openai/gpt-4.1-mini")
embedding = OpenAIEmbedding(
    model_name="text-embedding-3-small",
    api_base="https://openrouter.ai/api/v1",
    api_key="OPENROUTER_API_KEY",
)

# Data prep

In [7]:
reader = WikipediaReader()
documents = reader.load_data(pages=["deep learning", "artificial inteligence"])

In [8]:
len(documents)

2

In [9]:
splitter = SemanticSplitterNodeParser(embed_model=embedding)
chunks = splitter.get_nodes_from_documents(documents)

In [10]:
len(chunks)

52

In [11]:
rprint(chunks[0].get_content())

In machine learning, deep learning (DL) focuses on utilizing multilayered neural networks to perform tasks such as 
classification, regression, and representation learning. The field takes inspiration from biological neuroscience 
and revolves around stacking artificial neurons into layers and "training" them to process data. The adjective 
"deep" refers to the use of multiple layers (ranging from three to several hundred or thousands) in the network. 
Methods used can be supervised, semi-supervised or unsupervised.
Some common deep learning network architectures include fully connected networks, deep belief networks, recurrent 
neural networks, convolutional neural networks, generative adversarial networks, transformers, and neural radiance 
fields. These architectures have been applied to fields including computer vision, speech recognition, natural 
language processing, machine translation, bioinformatics, drug design, medical image analysis, climate science, 
material inspection and board game programs, where they have produced results comparable to and in some cases 
surpassing human expert performance.
Early forms of neural networks were inspired by information processing and distributed communication nodes in 
biological systems, particularly the human brain. However, current neural networks do not intend to model the brain
function of organisms, and are generally seen as low-quality models for that purpose.


== Overview ==
Most modern deep learning models are based on multi-layered neural networks such as convolutional neural networks 
and transformers, although they can also include propositional formulas or latent variables organized layer-wise in
deep generative models such as the nodes in deep belief networks and deep Boltzmann machines.
Fundamentally, deep learning refers to a class of machine learning algorithms in which a hierarchy of layers is 
used to transform input data into a progressively more abstract and composite representation. For example, in an 
image recognition model, the raw input may be an image (represented as a tensor of pixels). The first 
representational layer may attempt to identify basic shapes such as lines and circles, the second layer may compose
and encode arrangements of edges, the third layer may encode a nose and eyes, and the fourth layer may recognize 
that the image contains a face.
Importantly, a deep learning process can learn which features to optimally place at which level on its own. Prior 
to deep learning, machine learning techniques often involved hand-crafted feature engineering to transform the data
into a more suitable representation for a classification algorithm to operate on. In the deep learning approach, 
features are not hand-crafted and the model discovers useful feature representations from the data automatically. 
This does not eliminate the need for hand-tuning; for example, varying numbers of layers and layer sizes can 
provide different degrees of abstraction.
The word "deep" in "deep learning" refers to the number of layers through which the data is transformed. More 
precisely, deep learning systems have a substantial credit assignment path (CAP) depth. The CAP is the chain of 
transformations from input to output. CAPs describe potentially causal connections between input and output. For a 
feedforward neural network, the depth of the CAPs is that of the network and is the number of hidden layers plus 
one (as the output layer is also parameterized). For recurrent neural networks, in which a signal may propagate 
through a layer more than once, the CAP depth is potentially unlimited. No universally agreed-upon threshold of 
depth divides shallow learning from deep learning, but most researchers agree that deep learning involves CAP depth
higher than two. CAP of depth two has been shown to be a universal approximator in the sense that it can emulate 
any function. Beyond that, more layers do not add to the function approximator ability of the network. De

# Graph store prep

> **Prerequisites — graph database connection**
>
> This section assumes you have a **Memgraph** instance running and reachable at the URL and port shown below (`bolt://localhost:7688`). If your instance runs on a different host or port, update the `url` parameter accordingly.
>
> **Swapping the backend:** OGRE-KG is not limited to Memgraph. You can replace `MemgraphPropertyGraphStore` with any LlamaIndex-compatible property-graph store, for example:
> - **Neo4j** — use `Neo4jPropertyGraphStore` from `llama_index.graph_stores.neo4j` and adjust the connection string to `bolt://localhost:7687` (Neo4j's default port).
> - **In-memory** — use `SimplePropertyGraphStore` from `llama_index.core.graph_stores` for quick experiments that require no external database.
>
> Note that some OGRE-KG features (e.g. `MemgraphCypherEntitySimilarityFinder`, full-text index builder) are backend-specific and will need their Memgraph-specific counterparts replaced with the equivalent Neo4j or in-memory implementations.

In [9]:
graph_store = MemgraphPropertyGraphStore(
    username="",
    password="",
    url="bolt://localhost:7688",
    database="memgraph",
)

graph_store.structured_query("MATCH (n) RETURN COUNT(n);")

[{'COUNT(n)': 423}]

In [10]:
recreate_index = False
extract_prompt = PromptTemplate(
    "Extract up to {max_knowledge_triplets} ontology-grounded triplets from the given text. "
    "Prefer relation labels that align with the provided ontology when possible.\n"
    "---------------------\n"
    "INITIAL ONTOLOGY:\n"
    "Entity Types: {allowed_entity_types}\n"
    "Relation Types: {allowed_relation_types}\n\n"
    "TEXT: {text}\n"
    "---------------------"
)
extractor = kg_ext.OntoDynamicLLMPathExtractor(
    llm=llm,
    extract_prompt=extract_prompt,
    max_triplets_per_chunk=5,
    num_workers=5,
)

if recreate_index:
    graph_store.structured_query("MATCH (n) DETACH DELETE n;")

    storage_ctx = StorageContext.from_defaults(property_graph_store=graph_store)
    index = PropertyGraphIndex(
        nodes=chunks,
        embed_model=embedding,
        kg_extractors=[extractor],
        storage_context=storage_ctx,
        show_progress=True,
    )
else:
    index = PropertyGraphIndex.from_existing(
        property_graph_store=graph_store,
        embed_model=embedding,
        kg_extractors=[extractor],
        show_progress=False,
    )

# Indexing

### Building full-text indices with `graph_db_utils`

`make_graph_text_index_builder(graph_store)` returns a backend-specific `GraphTextIndexBuilder`.  
Calling `create_indices()` creates two full-text indices (if they don't already exist):
- **`entity_name`** on `__Entity__.name` — used by keyword entity retrievers to seed graph traversal.
- **`chunk_text`** on `Chunk.text` — used by chunk-keyword retrievers to find relevant source text chunks.

These indices are **required** before using any `MemgraphKeywordContextRetriever` or `MemgraphChunkKeywordRetriever`.

In [11]:
builder = kg_db_utils.make_graph_text_index_builder(graph_store)
builder.create_indices()

# Similar entities finding

### Finding similar entities with `MemgraphCypherEntitySimilarityFinder`

`MemgraphCypherEntitySimilarityFinder` leverages Memgraph's built-in `node_similarity.cosine` procedure.  
It projects a subgraph of `__Entity__` nodes and their relationships, computes pairwise cosine similarity on the pre-stored `embedding` property, and returns pairs above the configured `similarity_threshold`.

The raw pairs are then grouped into **connected components** (union-find) so that transitively similar entities end up in the same set — this avoids creating redundant, partially overlapping groups in later merging steps.

In [12]:
sim_finder = kg_sim.MemgraphCypherEntitySimilarityFinder(
    graph_store, embedding_attr="embedding", similarity_threshold=0.95
)
similar_ents = sim_finder.find_similar_entities()
len(similar_ents)

0

In [16]:
similar_ents

[]

### Merging duplicates with `MemgraphEntityMerger` + `EntityDisambiguationProcessor`

Once similar-entity groups are known, two OGRE-KG components handle the actual graph modification:

- **`MemgraphEntityMerger`** — wraps Memgraph's `refactor.merge_nodes` MAGE procedure.  
  It accepts a `merging_strategy` (`override`, `combine`, or `discard`) that controls how conflicting node properties are resolved, and a `merge_relations` flag to also consolidate duplicate relationships.  
  Setting `preview_changes=True` lets you inspect the generated Cypher queries without touching the graph.

- **`EntityDisambiguationProcessor`** — acts as a *coordinator*: it calls `similarity_finder.find_similar_entities()` and pipes the result directly to `merger.merge_entities()`.  
  This separation of concerns means you can swap in any other finder (e.g. `FuzzyEntitySimilarityFinder`) or merger (e.g. `MemgraphSynonymCreator`, `Neo4jEntityMerger`) without changing the orchestration logic.

In [ ]:
merger = kg_merger.MemgraphEntityMerger(graph_store, preview_changes=False)
disambiguator = kg_disamb.EntityDisambiguationProcessor(sim_finder, merger)

merged_ents = disambiguator.process()

# Extraction

### Custom retrievers from `memgraph_retrievers`

OGRE-KG provides two LlamaIndex-compatible sub-retrievers that use Memgraph full-text search:

- **`MemgraphKeywordContextRetriever`** — extracts keywords from the query via the LLM, then searches the `entity_name` full-text index to find matching entity nodes. From each seed entity it walks outgoing relationships in the graph and returns the surrounding context as `NodeWithScore` objects.

- **`MemgraphChunkKeywordRetriever`** — searches the `chunk_text` full-text index for document chunks whose text matches query keywords. When `restrict_to_seed_chunks=True` it limits traversal to chunks already linked to the seed entities found by the keyword search, keeping results tightly focused.

Both retrievers plug directly into a `PropertyGraphIndex.as_query_engine(sub_retrievers=[...])` call, replacing (or supplementing) the default vector and synonym retrievers supplied by LlamaIndex.

In [13]:
keyword_prompt = PromptTemplate(
    "Generate up to {max_keywords} graph lookup keywords for the user question.\n"
    "Return only concise entity names or graph terms.\n"
    "QUESTION: {question}"
)
chunk_terms_prompt = PromptTemplate(
    "Generate up to {max_terms} concise chunk fulltext search phrases for the user question.\n"
    "Prefer phrases that are likely to appear verbatim in the source text.\n"
    "QUESTION: {question}"
)

memgraph_kw_retriever = mg_ret.MemgraphKeywordContextRetriever(
    graph_store,
    llm=llm,
    topk_search=3,
    higher_score_is_better=True,
    keyword_prompt=keyword_prompt,
)
chunk_search = mg_ret.MemgraphChunkKeywordRetriever(
    graph_store,
    llm=llm,
    topk_search=3,
    max_chunk_terms=2,
    restrict_to_seed_chunks=True,
    higher_score_is_better=True,
    keyword_prompt=keyword_prompt,
    chunk_terms_prompt=chunk_terms_prompt,
)

In [14]:
question = "What is a vanishing gradient problem?"

In [17]:
kw_nodes = memgraph_kw_retriever.retrieve(question)
chunk_nodes = chunk_search.retrieve(question)

In [18]:
len(kw_nodes), len(chunk_nodes)

(4, 6)

In [19]:
rprint(kw_nodes[0].get_content())

Here are some facts extracted from the provided text:

Sepp Hochreiter -> IDENTIFIED -> vanishing gradient problem

Recurrence is used for sequence processing, and when a recurrent network is unrolled, it mathematically resembles a
deep feedforward layer. Consequently, they have similar properties and issues, and their developments had mutual 
influences. In RNN, two early influential works were the Jordan network (1986) and the Elman network (1990), which 
applied RNN to study problems in cognitive psychology.
In the 1980s, backpropagation did not work well for deep learning with long credit assignment paths. To overcome 
this problem, in 1991, Jürgen Schmidhuber proposed a hierarchy of RNNs pre-trained one level at a time by 
self-supervised learning where each RNN tries to predict its own next input, which is the next unexpected input of 
the RNN below. This "neural history compressor" uses predictive coding  to learn internal representations at 
multiple self-organizing time scales. This can substantially facilitate downstream deep learning. The RNN hierarchy
can be collapsed into a single RNN, by  distilling a higher level chunker network into a lower level automatizer 
network. In 1993, a neural history compressor solved a "Very Deep Learning" task that required more than 1000 
subsequent layers in an RNN unfolded in time. The "P" in ChatGPT refers to such pre-training.
Sepp Hochreiter's diploma thesis (1991) implemented the neural history compressor, and identified and analyzed the 
vanishing gradient problem.

In [21]:
qe = index.as_query_engine(sub_retrievers=[chunk_search, memgraph_kw_retriever], llm=llm)

In [22]:
res = qe.query(question)

In [24]:
rprint(res.response)

The vanishing gradient problem is a training difficulty that occurs when using gradient-based learning (e.g., 
backpropagation) in very deep networks or recurrent networks unrolled over many time steps. As error gradients are 
propagated backward through many layers or time steps, they can shrink (often exponentially), so that weights in 
early layers or at long time lags receive negligibly small updates. The result is that the model cannot effectively
learn long-range or long-term dependencies. Architectures and techniques such as recurrent residual connections and
long short-term memory (LSTM) units were developed to mitigate this problem.